# L0: Data Ingestion (Landing → Bronze)
## Enterprise-Grade Databricks ETL
---
**Purpose**: Load data from multiple sources (S3, HTTP, ADLS) into the Bronze layer (L0)

**Supported Formats**: CSV, JSON, Parquet, Excel, XML, Avro, ORC, Delta

**Control Table**: `demo_catalog.admin.data_flow_l0_detail`

**Layer**: L0 (Bronze/Landing)

---

## 1. Configuration & Parameters

In [ ]:
# ════════════════════════════════════════════════════════════════
# WIDGET PARAMETERS
# ════════════════════════════════════════════════════════════════

dbutils.widgets.text("DATA_FLOW_GROUP_ID", "", "Data Flow Group ID (e.g., EMPLOYEE_MASTER_L0)")
dbutils.widgets.text("ENVIRONMENT", "dev", "Environment: dev/qa/prod")
dbutils.widgets.text("LOB", "", "Line of Business filter (optional)")
dbutils.widgets.dropdown("RUN_MODE", "SYNC", ["SYNC", "ASYNC"], "SYNC=wait for completion, ASYNC=fire and forget")

# Get parameters
DATA_FLOW_GROUP_ID = dbutils.widgets.get("DATA_FLOW_GROUP_ID").strip().upper()
ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT").strip().lower()
LOB = dbutils.widgets.get("LOB").strip().upper()
RUN_MODE = dbutils.widgets.get("RUN_MODE").strip().upper()

# Validate
if not DATA_FLOW_GROUP_ID:
    raise ValueError("DATA_FLOW_GROUP_ID is mandatory")

if ENVIRONMENT not in ["dev", "qa", "prod"]:
    raise ValueError("ENVIRONMENT must be: dev, qa, or prod")

print(f"╔══════════════════════════════════════════════════════════╗")
print(f"║ L0 DATA INGESTION - EXECUTION PARAMETERS                 ║")
print(f"╠══════════════════════════════════════════════════════════╣")
print(f"║ Data Flow Group ID : {DATA_FLOW_GROUP_ID:<39} ║")
print(f"║ Environment        : {ENVIRONMENT:<39} ║")
print(f"║ LOB Filter         : {LOB or '(All)':<39} ║")
print(f"║ Run Mode           : {RUN_MODE:<39} ║")
print(f"╚══════════════════════════════════════════════════════════╝")

In [ ]:
# ════════════════════════════════════════════════════════════════
# IMPORT LIBRARIES
# ════════════════════════════════════════════════════════════════

import json
import traceback
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Databricks configuration
CATALOG = "demo_catalog"
CONTROL_SCHEMA = "admin"
BRONZE_SCHEMA = "bronze"
AUDIT_TABLE = f"{CATALOG}.{CONTROL_SCHEMA}.audit_log"

print(f"✓ Catalog configured: {CATALOG}")
print(f"✓ Control schema: {CONTROL_SCHEMA}")
print(f"✓ Bronze schema: {BRONZE_SCHEMA}")

## 2. Utility Functions

In [ ]:
# ════════════════════════════════════════════════════════════════
# FUNCTION: Log audit entry
# ════════════════════════════════════════════════════════════════

def log_audit(status, target_table, rows_count=0, error_msg=None, start_time=None, end_time=None):
    """
    Write audit log entry to track L0 execution
    
    Args:
        status: 'SUCCESS', 'FAILED', 'RUNNING'
        target_table: target table name
        rows_count: rows processed
        error_msg: error message (if failed)
        start_time: execution start time
        end_time: execution end time
    """
    try:
        audit_df = spark.createDataFrame([
            {
                "DATA_FLOW_GROUP_ID": DATA_FLOW_GROUP_ID,
                "TARGET_TABLE": target_table,
                "STATUS": status,
                "MESSAGE": error_msg or f"L0 ingestion {status.lower()}",
                "CREATED_DATE": datetime.now(),
                "ETL_LAYER": "L0",
                "ROWS_PROCESSED": rows_count,
                "START_TIME": start_time or datetime.now(),
                "END_TIME": end_time or datetime.now(),
                "LOAD_TS": datetime.now()
            }
        ])
        
        audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)
        print(f"✓ Audit logged: {status} - {target_table}")
    except Exception as e:
        print(f"⚠ Failed to write audit log: {str(e)}")

def log_execution_start():
    return datetime.now()

def log_execution_end(start_time):
    elapsed = (datetime.now() - start_time).total_seconds()
    return datetime.now(), elapsed

print("✓ Audit logging functions initialized")

In [ ]:
# ════════════════════════════════════════════════════════════════
# FUNCTION: Read source data (HTTP/S3/ADLS/DBFS/Volume)
# ════════════════════════════════════════════════════════════════

def read_source(source_url, file_format, delimiter=",", custom_schema=None):
    """
    Read data from various sources into a Spark DataFrame
    
    Args:
        source_url: URL/path to source (http/https/s3/adls/dbfs/volumes)
        file_format: csv, json, parquet, excel, xml, avro, orc, delta
        delimiter: delimiter for CSV files
        custom_schema: StructType for custom schema
    
    Returns:
        Spark DataFrame
    """
    file_format = file_format.strip().lower()
    source_url = source_url.strip()
    
    print(f"\n📥 Reading [{file_format.upper()}] from: {source_url[:100]}...")
    start_read = datetime.now()
    
    try:
        # HTTP/HTTPS sources
        if source_url.startswith(("http://", "https://")):
            import requests
            import io
            
            response = requests.get(source_url, timeout=120)
            response.raise_for_status()
            
            if file_format == "csv":
                pdf = pd.read_csv(io.StringIO(response.text), delimiter=delimiter)
            elif file_format == "json":
                pdf = pd.read_json(io.StringIO(response.text))
            elif file_format == "excel":
                pdf = pd.read_excel(io.BytesIO(response.content))
            else:
                raise ValueError(f"Format {file_format} not supported for HTTP sources")
            
            df = spark.createDataFrame(pdf)
        
        # S3/ADLS/DBFS/Delta/Parquet
        else:
            if file_format == "csv":
                df = spark.read.option("delimiter", delimiter).option("header", "true").csv(source_url)
            elif file_format == "json":
                df = spark.read.json(source_url)
            elif file_format == "parquet":
                df = spark.read.parquet(source_url)
            elif file_format == "delta":
                df = spark.read.format("delta").load(source_url)
            elif file_format == "avro":
                df = spark.read.format("avro").load(source_url)
            elif file_format == "orc":
                df = spark.read.orc(source_url)
            elif file_format == "excel":
                # Use pandas for Excel
                pdf = pd.read_excel(source_url)
                df = spark.createDataFrame(pdf)
            else:
                raise ValueError(f"Unsupported format: {file_format}")
        
        elapsed = (datetime.now() - start_read).total_seconds()
        row_count = df.count()
        print(f"✓ Read {row_count:,} rows in {elapsed:.2f}s")
        print(f"✓ Schema: {len(df.columns)} columns")
        
        return df, row_count
    
    except Exception as e:
        print(f"✗ Failed to read source: {str(e)}")
        raise

print("✓ Source reader function initialized")

In [ ]:
# ════════════════════════════════════════════════════════════════
# FUNCTION: Apply data quality checks
# ════════════════════════════════════════════════════════════════

def apply_dq_checks(df, dq_logic_json=None):
    """
    Apply data quality validations
    
    Args:
        df: Input DataFrame
        dq_logic_json: JSON string with DQ rules, e.g.
            '{"null_check": ["col1", "col2"], "unique_check": ["id"]}'
    
    Returns:
        DataFrame with DQ validation columns added
    """
    if not dq_logic_json:
        print("ℹ No DQ logic specified, skipping quality checks")
        return df
    
    print("\n🔍 Applying Data Quality Checks...")
    
    try:
        dq_rules = json.loads(dq_logic_json)
        
        # Add DQ validation column
        df = df.withColumn("DQ_VALIDATION_PASSED", F.lit(True))
        
        # Null checks
        if "null_check" in dq_rules:
            for col in dq_rules["null_check"]:
                if col in df.columns:
                    df = df.withColumn(
                        "DQ_VALIDATION_PASSED",
                        F.when(F.col(col).isNull(), F.lit(False)).otherwise(F.col("DQ_VALIDATION_PASSED"))
                    )
                    print(f"  ✓ Null check: {col}")
        
        # Add ingestion timestamp
        df = df.withColumn("INGESTION_TIMESTAMP", F.current_timestamp())
        
        print(f"✓ DQ checks applied successfully")
        return df
    
    except Exception as e:
        print(f"⚠ DQ check failed: {str(e)}. Continuing without DQ.")
        return df.withColumn("INGESTION_TIMESTAMP", F.current_timestamp())

print("✓ Data quality function initialized")

In [ ]:
# ════════════════════════════════════════════════════════════════
# FUNCTION: Write to Bronze layer
# ════════════════════════════════════════════════════════════════

def write_to_bronze(df, target_schema, target_table, load_type="FULL", partition_cols=None):
    """
    Write DataFrame to Bronze layer table
    
    Args:
        df: Input DataFrame
        target_schema: Target schema name
        target_table: Target table name
        load_type: 'FULL' (overwrite) or 'DELTA' (append/merge)
        partition_cols: List of partition columns
    """
    full_table_name = f"{CATALOG}.{target_schema}.{target_table}"
    print(f"\n📤 Writing to Bronze: {full_table_name}")
    print(f"  Load Type: {load_type}")
    
    try:
        start_write = datetime.now()
        row_count = df.count()
        
        # Determine write mode
        write_mode = "overwrite" if load_type.upper() == "FULL" else "append"
        
        writer = df.write.format("delta").mode(write_mode)
        
        # Add partitioning if specified
        if partition_cols:
            writer = writer.partitionBy(partition_cols)
            print(f"  Partitioning by: {', '.join(partition_cols)}")
        
        writer.option("mergeSchema", "true").saveAsTable(full_table_name)
        
        elapsed = (datetime.now() - start_write).total_seconds()
        print(f"✓ Successfully wrote {row_count:,} rows in {elapsed:.2f}s")
        
        return row_count, None
    
    except Exception as e:
        error_msg = f"Failed to write to Bronze: {str(e)}"
        print(f"✗ {error_msg}")
        return 0, error_msg

print("✓ Bronze writer function initialized")

## 3. Main L0 Execution Logic

In [ ]:
# ════════════════════════════════════════════════════════════════
# READ L0 CONFIGURATION FROM CONTROL TABLE
# ════════════════════════════════════════════════════════════════

control_table = f"{CATALOG}.{CONTROL_SCHEMA}.data_flow_l0_detail"
print(f"\n🔍 Reading L0 configuration from: {control_table}")

# Build query filter
where_clause = f"DATA_FLOW_GROUP_ID = '{DATA_FLOW_GROUP_ID}' AND IS_ACTIVE = 'Y'"
if LOB:
    where_clause += f" AND LOB = '{LOB}'"

print(f"Filter: {where_clause}")

try:
    config_df = spark.sql(f"SELECT * FROM {control_table} WHERE {where_clause}")
    config_count = config_df.count()
    
    if config_count == 0:
        raise ValueError(f"No active L0 configurations found for: {DATA_FLOW_GROUP_ID}")
    
    print(f"✓ Found {config_count} active L0 configuration(s)")
    config_df.display()
    
except Exception as e:
    print(f"✗ Failed to read control table: {str(e)}")
    raise

In [ ]:
# ════════════════════════════════════════════════════════════════
# EXECUTE L0 INGESTION FOR EACH CONFIGURATION
# ════════════════════════════════════════════════════════════════

execution_start = log_execution_start()
total_rows_loaded = 0
failed_tables = []
successful_tables = []

print("\n" + "="*70)
print("L0 INGESTION EXECUTION SUMMARY")
print("="*70)

for row in config_df.collect():
    try:
        source_url = row.SOURCE
        file_format = row.INPUT_FILE_FORMAT or "csv"
        delimiter = row.DELIMETER or ","
        target_schema = "bronze"
        target_table = row.SOURCE_OBJ_NAME.replace(".csv", "").replace(".json", "").lower()
        load_type = row.LOAD_TYPE or "FULL"
        dq_logic = row.DQ_LOGIC
        partition_cols = row.PARTITION.split(",") if row.PARTITION else None
        
        print(f"\n[{DATA_FLOW_GROUP_ID}] Loading: {target_table}")
        
        # Step 1: Read source
        df, row_count = read_source(source_url, file_format, delimiter)
        total_rows_loaded += row_count
        
        # Step 2: Apply DQ checks
        df = apply_dq_checks(df, dq_logic)
        
        # Step 3: Write to Bronze
        rows_written, error = write_to_bronze(df, target_schema, target_table, load_type, partition_cols)
        
        if error:
            failed_tables.append((target_table, error))
            log_audit("FAILED", target_table, 0, error, execution_start)
        else:
            successful_tables.append(target_table)
            log_audit("SUCCESS", target_table, rows_written, None, execution_start)
    
    except Exception as e:
        error_msg = f"{str(e)}\n{traceback.format_exc()}"
        print(f"✗ Error: {error_msg}")
        failed_tables.append((target_table, error_msg))
        log_audit("FAILED", target_table, 0, error_msg, execution_start)

# Print summary
execution_end, elapsed_secs = log_execution_end(execution_start)

print("\n" + "="*70)
print(f"EXECUTION SUMMARY - Total Time: {elapsed_secs:.2f}s")
print("="*70)
print(f"✓ Successful: {len(successful_tables)} table(s)")
for table in successful_tables:
    print(f"  • {table}")

if failed_tables:
    print(f"\n✗ Failed: {len(failed_tables)} table(s)")
    for table, error in failed_tables:
        print(f"  • {table}: {error[:100]}...")

print(f"\nTotal Rows Loaded: {total_rows_loaded:,}")
print(f"Completed at: {execution_end.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

## 4. Validation & Quality Report

In [ ]:
# ════════════════════════════════════════════════════════════════
# DATA QUALITY REPORT
# ════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("DATA QUALITY VALIDATION REPORT")
print("="*70)

for table in successful_tables:
    try:
        full_table = f"{CATALOG}.bronze.{table}"
        df = spark.table(full_table)
        
        print(f"\n📊 Table: {table}")
        print(f"  Rows: {df.count():,}")
        print(f"  Columns: {len(df.columns)}")
        
        # Show null counts
        null_counts = df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns])
        null_counts_dict = null_counts.collect()[0].asDict()
        
        null_cols = {k: v for k, v in null_counts_dict.items() if v > 0}
        if null_cols:
            print(f"  ⚠ Null values detected:")
            for col, count in null_cols.items():
                print(f"    - {col}: {count} nulls")
        else:
            print(f"  ✓ No null values detected")
    
    except Exception as e:
        print(f"  ⚠ Could not validate: {str(e)}")

print("\n" + "="*70)